# HW14 — Эмбеддинги, FAISS, оценка Retrieval и mini-RAG

**Тема:** Векторные представления текста, индекс FAISS, оценка качества retrieval, обновление базы знаний, mini-RAG.

**База знаний:** 15 документов по языку Python (история, синтаксис, типы данных, ООП, и т.д.)

**Структура ноутбука:**
1. Импорты, seed и среда
2. База знаний и первичный анализ
3. Чанкинг документов
4. Эмбеддинги и индекс FAISS
5. Контрольные запросы и оценка retrieval
6. Эксперимент с параметрами retrieval
7. Обновление базы знаний и переиндексация
8. Mini-RAG
9. Краткий анализ ошибок


## 1. Импорты, seed и среда

In [ ]:
# ─── Установка зависимостей (Python 3.11.3 + CUDA 13) ─────────────────────────
# [FIX] faiss-gpu не поддерживается через PyPI начиная с v1.7.3.
#       Для CUDA 12/13 используем faiss-gpu-cu12 (пакет с поддержкой CUDA 12.x+).
#       Если предпочтителен CPU-only режим — заменить на: pip install faiss-cpu
# [FIX] scikit-learn >= 1.3 полностью поддерживает Python 3.11
# [FIX] numpy < 2.0 для совместимости с faiss-gpu-cu12 (он собран против numpy 1.x ABI)
# [FIX] matplotlib >= 3.8 поддерживает Python 3.11
import subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import faiss
    print(f"faiss уже установлен: {faiss.__version__}")
except ImportError:
    # Пробуем GPU-пакет (требует CUDA Runtime 12.x+, совместим с CUDA 13)
    try:
        _pip("faiss-gpu-cu12")
        import faiss
        print("faiss-gpu-cu12 установлен успешно")
    except Exception:
        # Фолбэк: CPU-only (не использует GPU, но полностью функциональен)
        _pip("faiss-cpu")
        import faiss
        print("Установлен faiss-cpu (CPU-only fallback)")

# [FIX] Проверяем, что numpy < 2.0 (faiss-gpu-cu12 несовместим с numpy 2.x ABI)
import numpy as np
major = int(np.__version__.split(".")[0])
if major >= 2:
    print(f"[WARN] numpy {np.__version__} может быть несовместим с faiss-gpu-cu12.")
    print("       Если возникает ошибка ImportError, выполни: pip install 'numpy<2.0'")


In [ ]:
# Стандартные библиотеки Python
import os        # работа с файловой системой и путями
import re        # регулярные выражения для разбиения текста на предложения
import random    # генератор случайных чисел (для фиксации seed)
import json      # сериализация/десериализация JSON для сохранения метрик

# Научные библиотеки
import numpy as np           # работа с матрицами и векторами эмбеддингов
import pandas as pd          # табличное представление результатов и сохранение CSV
import matplotlib
import matplotlib.pyplot as plt  # построение графиков качества retrieval

# [FIX] matplotlib >= 3.8: явно задаём backend до создания фигур,
#       чтобы избежать предупреждений в headless-среде (Jupyter без GUI)
matplotlib.use("Agg")          # non-interactive backend для серверов без дисплея
matplotlib.rcParams["figure.dpi"] = 100

# Библиотека для векторного поиска (уже импортирована в ячейке выше)
import faiss  # индекс FAISS для быстрого поиска ближайших соседей

# TF-IDF векторизатор из scikit-learn
# [FIX] sklearn >= 1.3 полностью совместим с Python 3.11.3
from sklearn.feature_extraction.text import TfidfVectorizer
# TfidfVectorizer: TF = частота слова в документе,
# IDF = log(N / df), sublinear_tf заменяет tf на 1+log(tf)

# --- Фиксируем seed для воспроизводимости ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Пути к папкам проекта ---
KB_DIR  = "./knowledge_base"         # исходная база знаний (15 документов)
KB_UPD  = "./knowledge_base_update"  # папка с новыми документами для обновления базы
ART_DIR = "./artifacts"              # папка для сохранения результатов
os.makedirs(ART_DIR, exist_ok=True)

print("Среда готова.")
print(f"faiss version  : {faiss.__version__}")
print(f"numpy version  : {np.__version__}")
print(f"pandas version : {pd.__version__}")
print(f"python version : {sys.version}")
print(f"SEED = {SEED}")

# [FIX] Проверяем доступность GPU через FAISS
# faiss.get_num_gpus() возвращает 0 если GPU недоступен или используется faiss-cpu
import sys
try:
    n_gpus = faiss.get_num_gpus()
    print(f"FAISS GPU устройств: {n_gpus}")
    if n_gpus == 0:
        print("  → GPU не обнаружен или используется faiss-cpu; индекс будет CPU-only.")
except AttributeError:
    # faiss-cpu не имеет get_num_gpus()
    print("FAISS GPU: не поддерживается (faiss-cpu)")


## 2. База знаний и первичный анализ

**Предметная область:** Документация по языку программирования Python (15 тематических статей).

**Почему это хорошая база для retrieval:** Каждый документ посвящён конкретной теме, поэтому запросы вида «как работают декораторы?» имеют однозначный правильный ответ.


In [ ]:
def load_docs(directory):
    """
    Загружает все .txt документы из указанной папки.

    Возвращает список словарей:
      - 'source': имя файла (идентификатор документа)
      - 'text':   содержимое файла (сырой текст)

    Файлы сортируются по имени для детерминированного порядка.
    """
    docs = []
    for fname in sorted(os.listdir(directory)):
        if fname.endswith(".txt"):
            fpath = os.path.join(directory, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                text = f.read().strip()
            docs.append({"source": fname, "text": text})
    return docs

docs = load_docs(KB_DIR)
print(f"Загружено документов: {len(docs)}")
print()

for d in docs[:5]:
    print(f"[{d['source']}]")
    print(d['text'][:200])
    print()


## 3. Чанкинг документов

Используем простой оконный чанкинг по предложениям.

**Параметры:**
- `chunk_size = 2` предложения на чанк
- `overlap = 1` предложение перекрытия


In [ ]:
def split_into_sentences(text):
    """
    Разбивает текст на список предложений через lookbehind по знакам конца предложения.

    Пример:
        'Python создан Гвидо. Первая версия — 1991 год.'
        → ['Python создан Гвидо.', 'Первая версия — 1991 год.']
    """
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if s.strip()]


def chunk_doc(doc, chunk_size=2, overlap=1):
    """
    Разбивает один документ на перекрывающиеся чанки.

    Параметры:
      chunk_size — предложений в чанке (по умолчанию 2)
      overlap    — предложений перекрытия между соседними чанками (по умолчанию 1)

    Почему overlap=1 важен: граничные предложения попадают в оба соседних чанка,
    что повышает recall при семантических запросах.
    """
    sents = split_into_sentences(doc["text"])
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(sents), step):
        chunk_sents = sents[i:i + chunk_size]
        if chunk_sents:
            chunks.append({
                "source":   doc["source"],
                "text":     " ".join(chunk_sents),
                "chunk_id": len(chunks)
            })
    return chunks


CHUNK_SIZE = 2
OVERLAP    = 1

all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_doc(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP))

print(f"Всего чанков: {len(all_chunks)}")
print()

print("=== Пример чанкинга doc_06_functions.txt ===")
for ch in all_chunks:
    if ch["source"] == "doc_06_functions.txt":
        print(f"  chunk {ch['chunk_id']}: {ch['text'][:120]}...")
        print()


## 4. Эмбеддинги и индекс FAISS

**Метод:** TF-IDF (sklearn) — простой baseline без внешних зависимостей.

**Индекс:** `faiss.IndexFlatIP` (косинусное сходство через нормализацию).

**top_k = 3** по умолчанию.


In [ ]:
chunk_texts = [ch["text"] for ch in all_chunks]

# --- TF-IDF эмбеддинги ---
# sublinear_tf=True: заменяет tf на 1+log(tf) — сглаживает влияние частых слов
# ngram_range=(1,2): уни- и биграммы увеличивают различимость схожих документов
tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)
X = tfidf.fit_transform(chunk_texts).toarray().astype("float32")

# --- Нормализация для косинусного сходства ---
# IndexFlatIP = Inner Product; при единичных векторах IP == cosine similarity
norms = np.linalg.norm(X, axis=1, keepdims=True)
norms[norms == 0] = 1.0  # защита от деления на ноль
X_norm = X / norms

DIM = X_norm.shape[1]
print(f"Размерность эмбеддинга: {DIM}")

# --- Строим FAISS индекс ---
# [FIX] IndexFlatIP — CPU-based, работает без GPU.
#       Для GPU-ускорения на CUDA 13 можно обернуть в faiss.index_cpu_to_gpu():
#
#   res = faiss.StandardGpuResources()
#   index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(DIM))
#
# Для учебной базы (< 200 векторов) CPU-индекс оптимален — нет оверхеда на transfer.
index = faiss.IndexFlatIP(DIM)
index.add(X_norm)
print(f"Векторов в индексе: {index.ntotal}")


In [ ]:
TOP_K = 3


def embed_query(query_text):
    """
    Преобразует текст запроса в нормализованный TF-IDF вектор.

    Важно: tfidf.transform() (не fit_transform!) — применяем обученный словарь,
    не переобучаем его.
    """
    q_vec = tfidf.transform([query_text]).toarray().astype("float32")
    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    return q_vec


def retrieve(query_text, k=TOP_K, idx=None, chunks=None):
    """
    Ищет top-k наиболее релевантных чанков для запроса.

    Параметры:
      query_text — текст запроса
      k          — количество результатов
      idx        — индекс FAISS (None → глобальный index)
      chunks     — список чанков (None → глобальный all_chunks)

    Возвращает список словарей {chunk_id, source, text, score},
    отсортированных по убыванию score.
    """
    if idx is None:
        idx = index
    if chunks is None:
        chunks = all_chunks

    q_vec = embed_query(query_text)
    scores, ids = idx.search(q_vec, k)

    results = []
    for score, i in zip(scores[0], ids[0]):
        results.append({
            "chunk_id": int(i),
            "source":   chunks[i]["source"],
            "text":     chunks[i]["text"],
            "score":    float(score)
        })
    return results


# --- Демонстрация поиска ---
demo_queries = [
    "как создавался Python",
    "что такое декоратор",
    "обработка исключений try except",
    "асинхронное программирование asyncio",
    "виртуальное окружение venv",
]

for q in demo_queries:
    res = retrieve(q)
    print(f"Запрос: '{q}'")
    for r in res:
        print(f"  [{r['source']}] score={r['score']:.3f}  {r['text'][:80]}...")
    print()


## 5. Контрольные запросы и оценка retrieval

**10 контрольных запросов** с известными правильными источниками.

Метрики:
- **hit@k** — доля запросов, где правильный источник вошёл в top-k
- **recall@k** — среднее: доля правильных источников в top-k
- **MRR** — Mean Reciprocal Rank = среднее 1/rank первого релевантного результата


In [ ]:
eval_queries = [
    {"query": "история создания Python Гвидо ван Россум",          "expected": "doc_01_history.txt"},
    {"query": "синтаксис отступов Python",                          "expected": "doc_02_syntax.txt"},
    {"query": "типы данных int float str bool",                     "expected": "doc_03_types.txt"},
    {"query": "список и кортеж отличия append",                    "expected": "doc_04_lists_tuples.txt"},
    {"query": "словарь ключ значение dict Python",                  "expected": "doc_05_dicts.txt"},
    {"query": "функция def return аргументы Python",                "expected": "doc_06_functions.txt"},
    {"query": "классы объекты наследование Python",                 "expected": "doc_07_oop.txt"},
    {"query": "обработка исключений try except finally",            "expected": "doc_08_exceptions.txt"},
    {"query": "генераторы yield Python",                            "expected": "doc_09_generators.txt"},
    {"query": "декоратор функция обёртка functools",                "expected": "doc_12_decorators.txt"},
]

rows = []
for item in eval_queries:
    results = retrieve(item["query"], k=TOP_K)
    retrieved_sources = [r["source"] for r in results]

    hit    = int(item["expected"] in retrieved_sources)
    recall = 1.0 if item["expected"] in retrieved_sources else 0.0

    # Ранг: позиция правильного документа в списке (1-based); None если не найден
    rank = None
    for rk, r in enumerate(results, 1):
        if r["source"] == item["expected"]:
            rank = rk
            break

    rows.append({
        "query":                   item["query"],
        "expected_source":         item["expected"],
        "retrieved_sources":       str(retrieved_sources),
        "hit_at_k":                hit,
        "recall_at_k":             recall,
        "rank_of_first_relevant":  rank,
    })

df_eval = pd.DataFrame(rows)
print(df_eval[["query", "expected_source", "hit_at_k", "rank_of_first_relevant"]].to_string())
print()

print(f"hit@{TOP_K}    = {df_eval['hit_at_k'].mean():.2f}")
print(f"recall@{TOP_K} = {df_eval['recall_at_k'].mean():.2f}")

# MRR = среднее 1/rank — штрафует за низкое место правильного документа
mrr = df_eval["rank_of_first_relevant"].dropna().apply(lambda x: 1/x).mean()
print(f"MRR@{TOP_K}    = {mrr:.2f}")


In [ ]:
df_eval.to_csv(f"{ART_DIR}/retrieval_eval.csv", index=False, encoding="utf-8")
print(f"Сохранено: {ART_DIR}/retrieval_eval.csv")


In [ ]:
# [FIX] matplotlib.use("Agg") уже задан выше — plt.show() работает в headless-режиме,
#       сохранение через savefig() функционирует корректно на Python 3.11.3

labels = ["Попал в top-k", "Не попал"]
counts = [int(df_eval['hit_at_k'].sum()), len(df_eval) - int(df_eval['hit_at_k'].sum())]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, counts, color=["#2ecc71", "#e74c3c"])
axes[0].set_title(f"Hit@{TOP_K} по контрольным запросам")
axes[0].set_ylabel("Кол-во запросов")
axes[0].set_ylim(0, 12)
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontsize=12)

ranks = df_eval['rank_of_first_relevant'].dropna().value_counts().sort_index()
axes[1].bar(ranks.index.astype(str), ranks.values, color="#3498db")
axes[1].set_title("Распределение ранга первого релевантного документа")
axes[1].set_xlabel("Ранг")
axes[1].set_ylabel("Кол-во запросов")
for i, (x, v) in enumerate(zip(ranks.index, ranks.values)):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(f"{ART_DIR}/retrieval_quality_plot.png", bbox_inches="tight")
plt.show()
print(f"График сохранён: {ART_DIR}/retrieval_quality_plot.png")


## 6. Эксперимент с параметрами retrieval

Сравниваем `k=1` vs `k=3`.


In [ ]:
# k=1 — строгий: успех только если правильный документ на 1-м месте
# k=3 — мягкий:  успех если правильный документ в первой тройке
results_exp = {}
for k_val in [1, 3]:
    hits = 0
    for item in eval_queries:
        retrieved = [r["source"] for r in retrieve(item["query"], k=k_val)]
        if item["expected"] in retrieved:
            hits += 1
    results_exp[f"top_{k_val}"] = {
        "k":        k_val,
        "hit_rate": hits / len(eval_queries),
        "hits":     hits,
        "total":    len(eval_queries)
    }

print("Сравнение top_k=1 vs top_k=3:")
print(f"  top_k=1: hit@1 = {results_exp['top_1']['hit_rate']:.2f}  ({results_exp['top_1']['hits']}/{results_exp['top_1']['total']})")
print(f"  top_k=3: hit@3 = {results_exp['top_3']['hit_rate']:.2f}  ({results_exp['top_3']['hits']}/{results_exp['top_3']['total']})")
print()
print("Вывод: при k=3 hit rate выше — больший k даёт больше кандидатов.")
print("Для mini-RAG оптимально k=3: достаточно контекста без перегрузки промпта.")

summary = {
    "experiment": "top_k comparison",
    "top_k_1":    results_exp['top_1'],
    "top_k_3":    results_exp['top_3'],
    "chosen_k":   3,
    "reason":     "k=3 даёт лучший hit rate при разумном объёме контекста"
}
with open(f"{ART_DIR}/retrieval_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Сохранено: {ART_DIR}/retrieval_metrics_summary.json")


## 7. Обновление базы знаний и переиндексация

Добавляем 5 новых документов из `knowledge_base_update/`:
- `doc_16_regex.txt`, `doc_17_json.txt`, `doc_18_comprehensions.txt`, `doc_19_fstrings.txt`, `doc_20_dataclasses.txt`


In [ ]:
# Сохраняем ссылки на «старый» индекс для сравнения до/после
index_before  = index
chunks_before = all_chunks.copy()
tfidf_before  = tfidf

def retrieve_before(query_text, k=TOP_K):
    return retrieve(query_text, k=k, idx=index_before, chunks=chunks_before)

# Загружаем новые документы
new_docs = load_docs(KB_UPD)
print(f"Новых документов: {len(new_docs)}")
for d in new_docs:
    print(f"  [{d['source']}] {d['text'][:80]}...")

# Чанкинг новых документов с теми же параметрами (важно для однородности базы)
new_chunks = []
for doc in new_docs:
    new_chunks.extend(chunk_doc(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP))
print(f"\nНовых чанков: {len(new_chunks)}")

# Объединяем старые и новые
all_docs_updated   = docs + new_docs
all_chunks_updated = chunks_before + new_chunks
chunk_texts_updated = [ch["text"] for ch in all_chunks_updated]

# --- Полное переобучение TF-IDF на расширенном корпусе ---
# Почему не инкрементальное: IDF-веса зависят от всего корпуса.
# Добавление документов меняет IDF старых слов → нужен полный перефит.
# [FIX] TfidfVectorizer в sklearn >= 1.3 корректно работает с Python 3.11.3
tfidf_updated = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)
X_updated = tfidf_updated.fit_transform(chunk_texts_updated).toarray().astype("float32")

norms_u = np.linalg.norm(X_updated, axis=1, keepdims=True)
norms_u[norms_u == 0] = 1.0
X_updated_norm = X_updated / norms_u

# [FIX] Новый индекс строится с новой размерностью DIM_NEW (словарь расширился)
DIM_NEW = X_updated_norm.shape[1]
index_updated = faiss.IndexFlatIP(DIM_NEW)
index_updated.add(X_updated_norm)

print(f"\nОбновлённый индекс: {index_updated.ntotal} векторов (DIM={DIM_NEW})")


def embed_query_updated(query_text):
    """Векторизует запрос через обновлённый TF-IDF (расширенный словарь)."""
    q_vec = tfidf_updated.transform([query_text]).toarray().astype("float32")
    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    return q_vec


def retrieve_updated(query_text, k=TOP_K):
    """Ищет top-k чанков в обновлённом индексе."""
    q_vec = embed_query_updated(query_text)
    scores, ids = index_updated.search(q_vec, k)
    results = []
    for score, i in zip(scores[0], ids[0]):
        results.append({
            "chunk_id": int(i),
            "source":   all_chunks_updated[i]["source"],
            "text":     all_chunks_updated[i]["text"],
            "score":    float(score)
        })
    return results


In [ ]:
compare_queries = [
    {"query": "регулярные выражения re.match Python",       "target_new": "doc_16_regex.txt"},
    {"query": "работа с JSON сериализация Python",          "target_new": "doc_17_json.txt"},
    {"query": "генераторы списков list comprehension",      "target_new": "doc_18_comprehensions.txt"},
    {"query": "f-строки форматирование Python 3.6",         "target_new": "doc_19_fstrings.txt"},
    {"query": "dataclass датакласс аннотации типов",        "target_new": "doc_20_dataclasses.txt"},
    {"query": "история создания Python Гвидо ван Россум",  "target_new": "doc_01_history.txt"},
]

ba_rows = []
for item in compare_queries:
    before_res = [r["source"] for r in retrieve(item["query"], k=TOP_K)]
    after_res  = [r["source"] for r in retrieve_updated(item["query"], k=TOP_K)]
    changed    = before_res != after_res
    ba_rows.append({
        "query":                    item["query"],
        "before_retrieved_sources": str(before_res),
        "after_retrieved_sources":  str(after_res),
        "changed":                  int(changed),
    })

df_ba = pd.DataFrame(ba_rows)
print(df_ba[["query", "before_retrieved_sources", "after_retrieved_sources", "changed"]].to_string())
print()
print(f"Изменилось результатов: {df_ba['changed'].sum()}/{len(df_ba)}")

df_ba.to_csv(f"{ART_DIR}/retrieval_before_after_update.csv", index=False, encoding="utf-8")
print(f"Сохранено: {ART_DIR}/retrieval_before_after_update.csv")


## 8. Mini-RAG

**Архитектура:**
1. Retrieval: top-k чанков через FAISS
2. Context assembly: объединение чанков в строку
3. Generation: шаблонный ответ (наиболее релевантный чанк)
4. Source attribution: список источников


In [ ]:
def mini_rag(question, k=TOP_K, use_updated=True):
    """
    Mini-RAG конвейер (Retrieval-Augmented Generation).

    Шаги:
      1. Retrieval — ищем top-k чанков через FAISS
      2. Context assembly — объединяем чанки в строку-контекст
      3. Generation — возвращаем наиболее релевантный чанк как ответ
         (в production здесь стоит LLM — GPT, Llama и т.д.)
      4. Source attribution — возвращаем список источников

    Параметры:
      use_updated — True: использовать обновлённый индекс
                    False: исходный индекс
    """
    if use_updated:
        results = retrieve_updated(question, k=k)
    else:
        results = retrieve(question, k=k)

    if not results:
        return {"question": question, "answer": "Информация не найдена.", "sources": []}

    # Контекст: все чанки с указанием источника
    context_parts = [f"[{r['source']}]: {r['text']}" for r in results]
    context = "\n\n".join(context_parts)

    # «Генерация»: берём текст лучшего чанка (наибольший cosine-score)
    best   = results[0]
    answer = (
        f"На основе документа '{best['source']}':\n"
        f"{best['text']}"
    )

    return {
        "question": question,
        "answer":   answer,
        "sources":  [r["source"] for r in results],
        "context":  context,
    }


rag_questions = [
    "Кто создал Python и когда?",
    "Чем список отличается от кортежа в Python?",
    "Как работает декоратор в Python?",
    "Как используется asyncio в Python?",
    "Как работают регулярные выражения в Python?",
    "Что такое dataclass?",
    "Как обрабатывать исключения в Python?",
]

rag_rows = []
for q in rag_questions:
    result = mini_rag(q)
    print(f"Вопрос: {result['question']}")
    print(f"Ответ:  {result['answer'][:200]}")
    print(f"Источники: {result['sources']}")
    print()
    rag_rows.append({
        "question":          result["question"],
        "answer":            result["answer"],
        "retrieved_sources": str(result["sources"]),
    })


In [ ]:
df_rag = pd.DataFrame(rag_rows)
df_rag.to_csv(f"{ART_DIR}/rag_examples.csv", index=False, encoding="utf-8")
print(f"Сохранено: {ART_DIR}/rag_examples.csv")
print()
print(df_rag[["question", "retrieved_sources"]].to_string())


## 9. Краткий анализ ошибок

Пограничные случаи, где TF-IDF retrieval может ошибиться.


In [ ]:
hard_cases = [
    "Как создать виртуальное окружение Python?",
    "Что такое yield?",
    "Как установить пакет pip?",
    "Как импортировать модуль?",
    "Что быстрее: list или tuple?",
]

print("=== Пограничные/проблемные случаи ===\n")
for q in hard_cases:
    result = mini_rag(q)
    print(f"Вопрос: {q}")
    print(f"Источник #1: {result['sources'][0] if result['sources'] else 'нет'}")
    print(f"Ответ (начало): {result['answer'][:150]}")
    print()

print("=== Анализ ошибок ===")
print("""
1. 'Как установить пакет pip?' — нет документа про pip → retrieval вернёт нерелевантный чанк.
   Причина: неполнота базы знаний. Решение: добавить doc_pip.txt.

2. 'Что такое yield?' — корректно находит doc_09_generators.txt:
   TF-IDF хорошо работает на точных терминах.

3. 'Как импортировать модуль?' — конкуренция doc_10_modules.txt и doc_13_virtualenv.txt.
   Причина: семантически близкие документы. Решение: dense embeddings (sentence-transformers).

4. 'Что быстрее: list или tuple?' — лексический разрыв: слово «быстрее» не в тексте документа.
   Фундаментальное ограничение TF-IDF: работает на совпадении слов, не на смысле.

5. Шаблонный генератор ограничен: возвращает готовый фрагмент, не синтезирует ответ.
   В реальном RAG здесь стоит LLM (GPT, Llama), которая формулирует связный ответ.
""")


## Итог

### Файлы артефактов:
- `artifacts/retrieval_eval.csv` — оценка retrieval на контрольных запросах
- `artifacts/rag_examples.csv` — примеры работы mini-RAG
- `artifacts/retrieval_before_after_update.csv` — сравнение до/после обновления базы
- `artifacts/retrieval_quality_plot.png` — график качества retrieval
- `artifacts/retrieval_metrics_summary.json` — сводка метрик

### Структура папки:
```
homeworks/HW14/
├── HW14.ipynb
├── report.md
├── knowledge_base/          (15 документов)
├── knowledge_base_update/   (5 новых документов)
└── artifacts/
    ├── retrieval_eval.csv
    ├── rag_examples.csv
    ├── retrieval_before_after_update.csv
    ├── retrieval_quality_plot.png
    └── retrieval_metrics_summary.json
```


In [ ]:
print("=== Все файлы артефактов ===")
for f in sorted(os.listdir(ART_DIR)):
    path = os.path.join(ART_DIR, f)
    size = os.path.getsize(path)
    print(f"  {f}  ({size} bytes)")

print("\n=== HW14 выполнено! ===")
